# 적용한 내용
[챗봇 실행 함수](#로그-저장이-포함된-챗봇-실행-함수-정의)에서 함수가 호출 되었을 때 대화 내역을 저장하는 리스트를 생성한다.
이후 일반 대화가 이루어 질 때(ask 함수가 호출될 때) 프롬프트와 응답을 리스트에 추가한다.
`/exit`이 입력된 경우 break 전 `chat_log.txt`에 리스트의 내용을 작성한다.


# 실행 환경 감지

In [1]:
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")


✅ 로컬 Jupyter


# API 키 로드

In [2]:
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")

✅ API 키 OK


# 패키지 임포트

In [10]:
# 패키지 import
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from datetime import datetime

print("✅ import 완료")

✅ import 완료


# llm, prompt, chain 생성 및 작동 확인

In [4]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role}입니다."),
    ("human", "{question}")
])

chain = prompt | llm   # LCEL 파이프 연산자 (2교시 3부)

# 동작 확인
result = chain.invoke({"role": "데이터 분석가", "question": "이상치란 무엇인가요? 한 문장으로."})
print(result.content)

이상치는 데이터 집합에서 다른 관측값들과 현저하게 다르거나 벗어난 값을 의미합니다.


# 페르소나 사전 생성

In [5]:
# 페르소나 사전: 이름 → system 메시지
PERSONAS = {
    "분석가": "당신은 10년 차 데이터 분석가입니다. 수치와 근거를 들어 간결하게 답변하세요.",
    "마케터": "당신은 디지털 마케팅 전문가입니다. 비즈니스 관점과 사례를 들어 답변하세요.",
    "개발자": "당신은 시니어 백엔드 개발자입니다. 코드 예시와 함께 실용적으로 답변하세요.",
    "기획자": "당신은 IT 서비스 기획자입니다. 사용자 가치와 우선순위 관점에서 답변하세요.",
    "디자이너": "당신은 UX 디자이너입니다. 사용자 경험과 시각적 비유로 쉽게 답변하세요.",
}

print(f"✅ {len(PERSONAS)}개 페르소나 등록 완료: {list(PERSONAS.keys())}")

✅ 5개 페르소나 등록 완료: ['분석가', '마케터', '개발자', '기획자', '디자이너']


# 챗봇 함수 정의

In [6]:
# 챗봇 핵심 함수: 페르소나 이름과 질문을 받아 답변 반환
def ask(persona_name: str, question: str) -> str:
    # 1. 페르소나 이름으로 system 메시지 조회
    system_message = PERSONAS[persona_name]

    # 2. 체인 호출 (2교시에서 배운 prompt | llm 패턴)
    result = chain.invoke({"role": system_message, "question": question})

    # 3. AIMessage 객체에서 답변 텍스트만 꺼내서 반환
    return result.content

In [7]:
# 명령어 처리 함수: (새로운_페르소나, 종료여부) 튜플 반환
def handle_command(user_input: str, current_persona: str):
    parts = user_input.split(maxsplit=1)   # "/change 마케터" → ["/change", "마케터"]
    cmd = parts[0]

    if cmd in ("/quit", "/exit"):
        print("👋 종료합니다.")
        return current_persona, True       # 종료 플래그 True

    if cmd == "/persona":
        print(f"🎭 현재 페르소나: {current_persona}")

    elif cmd == "/help":
        print("📖 명령어: /quit, /exit, /change <이름>, /persona, /help")

    elif cmd == "/change":
        new_name = parts[1] if len(parts) > 1 else ""
        if new_name in PERSONAS:
            print(f"✅ 페르소나가 '{new_name}'(으)로 변경되었습니다.")
            return new_name, False         # 페르소나만 변경
        else:
            print(f"❌ '{new_name}'은 등록된 페르소나가 아닙니다.")
            print(f"   사용 가능: {', '.join(PERSONAS.keys())}")

    else:
        print(f"❓ 알 수 없는 명령어: {cmd} (/help 입력)")

    return current_persona, False

# 로그 저장이 포함된 챗봇 실행 함수 정의

In [11]:
# 챗봇 실행 함수
def run_chatbot(initial_persona: str = "분석가"):
    # 대화 누적 리스트
    log = list()

    persona = initial_persona
    print(f"👋 페르소나 챗봇입니다. (현재: {persona})")
    print("   /help 로 명령어 보기, /quit 로 종료\n")

    while True:
        user_input = input("질문> ").strip()
        if not user_input:                       # 빈 입력은 무시
            continue

        if user_input.startswith("/"):           # 명령어 처리
            persona, should_quit = handle_command(user_input, persona)
            if should_quit:
                # 챗봇 종료 시 로그 리스트를 로그 파일로 저장하고 종료
                with open("chat_log.txt", "w", encoding="utf-8") as f:
                    f.write("\n".join(log))
                break
        else:                                    # 일반 질문 처리
            answer = ask(persona, user_input)
            print(f"[{persona}] {answer}\n")
            log.append(f"[{datetime.now().strftime('%H:%M:%S')}] [{persona}] {user_input} → {answer}") # 일반 질문과 응답을 로그 리스트에 추

# 챗봇 실행

In [ ]:
run_chatbot()

👋 페르소나 챗봇입니다. (현재: 분석가)
   /help 로 명령어 보기, /quit 로 종료

[분석가] 페르소나(Persona)는 특정 사용자나 고객의 특성을 반영한 가상의 인물입니다. 주로 마케팅, UX/UI 디자인, 제품 개발 등에서 사용되며, 목표 고객의 행동, 필요, 동기 등을 이해하는 데 도움을 줍니다. 

페르소나를 만들기 위해서는 다음과 같은 요소를 고려합니다:

1. **인구 통계학적 정보**: 나이, 성별, 직업, 교육 수준 등.
2. **행동 패턴**: 구매 습관, 온라인 활동, 브랜드 선호도 등.
3. **목표와 동기**: 제품이나 서비스 사용의 이유, 해결하고자 하는 문제 등.
4. **고충과 장애물**: 제품 사용 시 겪는 어려움이나 불만 사항.

이러한 페르소나는 실제 데이터(설문조사, 인터뷰, 웹 분석 등)를 기반으로 하여 만들어지며, 기업이 고객의 요구를 더 잘 이해하고 맞춤형 전략을 수립하는 데 기여합니다.

✅ 페르소나가 '기획자'(으)로 변경되었습니다.
[기획자] 페르소나(Persona)는 특정 사용자 그룹의 특성과 행동을 대표하는 가상의 인물입니다. IT 서비스 기획 및 디자인 과정에서 페르소나는 사용자 경험(UX)을 개선하고, 제품이나 서비스의 요구사항을 명확히 이해하는 데 중요한 역할을 합니다. 

페르소나를 정의할 때는 다음과 같은 요소를 고려합니다:

1. **인구통계학적 정보**: 나이, 성별, 직업, 교육 수준 등.
2. **행동 패턴**: 사용자가 서비스를 어떻게 이용하는지, 어떤 문제를 겪는지, 어떤 목표를 가지고 있는지.
3. **니즈와 목표**: 사용자가 서비스에서 무엇을 원하는지, 어떤 가치를 추구하는지.
4. **고통점**: 사용자가 겪고 있는 문제나 불편함.

페르소나를 활용하면 다음과 같은 사용자 가치를 창출할 수 있습니다:

- **사용자 중심의 설계**: 페르소나를 통해 실제 사용자의 요구와 기대를 반영한 제품이나 서비스를 설계할 수 있습니다.
- **우선순위 설정**: 다양한 사용자 요구를

: 